In [4]:
import os
import json
from itertools import combinations
from collections import Counter


# ==========================================================
# Dataset location
# ==========================================================

base_path = "/Users/shergoshami/Downloads/DATA"


folders = {
    "Female Neutral": "female_neutral",
    "Female ProChoice": "female_prochoice",
    "Female ProLife": "female_prolife",
    "Male Neutral": "male_neutral",
    "Male ProChoice": "male_prochoice",
    "Male ProLife": "male_prolife"
}



# ==========================================================
# Number of first comments
# ==========================================================

N = 5



# ==========================================================
# Load dataset
#
# group_files[group][filename]
#
# ==========================================================

group_files = {}



for group, folder in folders.items():

    folder_path = os.path.join(
        base_path,
        folder
    )


    if not os.path.exists(folder_path):

        print(
            f"Missing folder: {folder_path}"
        )

        continue



    group_files[group] = {}



    for file in sorted(os.listdir(folder_path)):


        if not file.endswith(".json"):

            continue



        file_path = os.path.join(
            folder_path,
            file
        )


        with open(
            file_path,
            "r",
            encoding="utf-8"
        ) as f:

            data = json.load(f)



        comments = []



        for comment in data.get(
            "comments",
            []
        ):


            text = comment.get(
                "comment",
                ""
            )


            # Normalize GIF comments

            if text.startswith("[GIF:"):

                text = "[GIF]"



            # Full comment signature

            comment_signature = {

                "username":
                    comment.get(
                        "username",
                        ""
                    ).strip(),


                "comment":
                    text,


                "likes":
                    comment.get(
                        "likes",
                        0
                    ),


                "replies":
                    comment.get(
                        "replies",
                        0
                    ),


                "is_verified":
                    comment.get(
                        "is_verified",
                        False
                    )

            }



            serialized_comment = json.dumps(
                comment_signature,
                sort_keys=True
            )


            comments.append(
                serialized_comment
            )





        # Store post information

        group_files[group][file] = {


            "post_id":
                data.get(
                    "url",
                    file
                ),


            "likes":
                data.get(
                    "post_total_likes",
                    0
                ),


            "comment_count":
                data.get(
                    "post_total_comments_count",
                    len(comments)
                ),


            "post_timestamp":
                data.get(
                    "post_timestamp",
                    ""
                ),


            "verified":
                data.get(
                    "post_publisher_is_verified",
                    False
                ),


            "comments":
                comments

        }





# ==========================================================
# First N comments
# ==========================================================

def first_n_comments(
        A,
        B,
        N
):

    return (

        A[:N],

        B[:N]

    )





# ==========================================================
# Divergence Formula
#
# D(A,B)=|A△B|/(|A|+|B|)
#
# Multiset version (duplicates preserved)
#
# ==========================================================

def divergence(
        A,
        B
):


    counter_A = Counter(A)

    counter_B = Counter(B)



    difference = (

        counter_A - counter_B

    ) + (

        counter_B - counter_A

    )



    numerator = sum(
        difference.values()
    )



    denominator = (

        len(A)

        +

        len(B)

    )



    if denominator == 0:

        return 0.0



    return numerator / denominator





# ==========================================================
# Extract gender and stance
# ==========================================================

def split_group(
        group_name
):

    parts = group_name.split()


    gender = parts[0]

    stance = parts[1]


    return gender, stance





# ==========================================================
# Find posts common to all groups
# ==========================================================

group_names = list(
    group_files.keys()
)



common_posts = set.intersection(

    *[

        set(
            group_files[g].keys()
        )

        for g in group_names

    ]

)



print(
    f"Common posts: {len(common_posts)}"
)





# ==========================================================
# Build Pairwise Divergence Dataset
#
# 15 comparisons per post
#
# ==========================================================

dataset = []



for filename in sorted(common_posts):


    for g1, g2 in combinations(
        group_names,
        2
    ):



        post_A = group_files[g1][filename]

        post_B = group_files[g2][filename]



        gender1, stance1 = split_group(g1)

        gender2, stance2 = split_group(g2)





        # --------------------------------------
        # First N comments
        # --------------------------------------

        A, B = first_n_comments(

            post_A["comments"],

            post_B["comments"],

            N

        )





        # --------------------------------------
        # Divergence
        # --------------------------------------

        d = divergence(

            A,

            B

        )





        dataset.append({

            "post_id":

                post_A["post_id"],


            "gender1":

                gender1,


            "gender2":

                gender2,


            "stance1":

                stance1,


            "stance2":

                stance2,


            "post_likes":

                post_A["likes"],


            "post_comments":

                post_A["comment_count"],


            "post_timestamp":

                post_A["post_timestamp"],


            "verified":

                post_A["verified"],


            "divergence":

                d

        })





# ==========================================================
# Save dataset
# ==========================================================

output_file = os.path.join(

    base_path,

    "post_divergence_dataset.json"

)



with open(

        output_file,

        "w",

        encoding="utf-8"

) as f:


    json.dump(

        dataset,

        f,

        indent=4,

        ensure_ascii=False

    )





print(
    f"\nSaved to: {output_file}"
)


print(
    f"Number of rows: {len(dataset)}"
)

Common posts: 612

Saved to: /Users/shergoshami/Downloads/DATA/post_divergence_dataset.json
Number of rows: 9180
